# Laboratorio 2 - AlpesHearth

## Integrantes

- Isabella Naranjo
- David Caro

## Conjunto de datos resultante del laboratorio 1

El conjunto de datos del laboratorio 1 paso por un proceso de exploración 

In [5]:
import pandas as pd 

df = pd.read_csv("./data/Datos Lab 1.csv")
df_model1 = df.copy()
df_model1 = df_model1.dropna(subset=["CVD Risk Score"])
before = len(df_model1)
df_model1 = df_model1.drop_duplicates(keep="first") #mantenemos solo la primera ocurrencia
after = len(df_model1)

print(f"Duplicados idénticos eliminados: {before - after}")

print("Antes:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
ids_duplicados = df_model1[df_model1["Patient ID"].duplicated(keep=False)]
df_model1 = df_model1[df_model1["CVD Risk Score"] >= 0]

cols_sin_target = df_model1.columns.tolist()
cols_sin_target.remove("CVD Risk Score")
cols_sin_target.remove("Patient ID")
df_base = df_model1.drop_duplicates(subset=["Patient ID"], keep="first")[["Patient ID"] + cols_sin_target]
df_score_prom = df_model1.groupby("Patient ID")["CVD Risk Score"].mean().reset_index()
df_model1 = df_base.merge(df_score_prom, on="Patient ID", how="left")

#verificacion
print("Después:", df_model1["Patient ID"].duplicated().sum(), "IDs duplicados")
print(df_model1.shape)

df_model1 = df_model1[df_model1["Age"] >= 18]
df_model1 = df_model1[df_model1["Weight (kg)"] >= 30]
df_model1 = df_model1[df_model1["BMI"] >= 10]
df_model1 = df_model1[df_model1["Estimated LDL (mg/dL)"] >= 0]

print("Dimensiones después de eliminar valores imposibles:")
print(df_model1.shape)

numeric_cols = df_model1.select_dtypes(include=["int64", "float64"]).columns
numeric_cols = [c for c in numeric_cols if c != "CVD Risk Score"]
print(numeric_cols)

def quitar_outliers(df, col):
    Q1 = df[col].quantile(0.25)   
    Q3 = df[col].quantile(0.75)   
    IQR = Q3 - Q1                 
    
    lower = Q1 - 1 * IQR        
    upper = Q3 + 1 * IQR        
    
    df_filtrado = df[(df[col] >= lower) & (df[col] <= upper)]
    
    print(f"{col}: antes={len(df)}, después={len(df_filtrado)}")
    return df_filtrado
#for col in numeric_cols:
   # df_model1 = quitar_outliers(df_model1, col)

print("Dimensiones finales después de quitar outliers:")
print(df_model1.shape)


Duplicados idénticos eliminados: 150
Antes: 111 IDs duplicados
Después: 0 IDs duplicados
(1344, 24)
Dimensiones después de eliminar valores imposibles:
(1119, 24)
['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Height (cm)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Dimensiones finales después de quitar outliers:
(1119, 24)


## Construcción de un modelo de regresión polinomial

In [6]:
# IMPORTS
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import FunctionTransformer, Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold

#PARTICION: sacar train y test para XY

target = "CVD Risk Score"
X = df_model1.drop(columns=[target])
y = df_model1[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=1
)

# Separacion de variables numericas y categoricas

numeric_features = ["Age",
                    "Weight (kg)","Height (m)",
                    "BMI",
                    "Abdominal Circumference (cm)",
                    "Total Cholesterol (mg/dL)",
                    "HDL (mg/dL)",
                    "Fasting Blood Sugar (mg/dL)",
                    "Waist-to-Height Ratio",
                    "Systolic BP",
                    "Diastolic BP",
                    "Estimated LDL (mg/dL)"
]

categorical_features = ["Sex","Smoking Status","Diabetes Status","Physical Activity Level",
                        "Family History of CVD","Blood Pressure Category"]

print("Numéricas:", list(numeric_features))
print("Categóricas:", list(categorical_features))

# Definir un dropper para eliminar columnas no deseadas 
columns_to_drop = [
    'Patient ID',
    'Blood Pressure (mmHg)',
    'Height (cm)',
    'CVD Risk Level',
    'Date of Service'
]

def drop_columns(df):
    return df.drop(columns=columns_to_drop, errors="ignore")

dropper = FunctionTransformer(drop_columns)

# PIPELINE: variables numericas
numeric_transforme_polinomial = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")), # imputa los valores faltantes con la media de cada columna
    ("scaler", StandardScaler()), # escala todas las variables numericas a la misma escala
    ("polynomial", PolynomialFeatures(degree=2)) # genera nuevas variables a partir de las originales usando grado 2
])

# PIPELINE: variables categoricas

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")), # imputa los valores faltantes con la moda de cada columna
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="if_binary")) # codifica las variables categoricas usando one-hot 
])

# COLUMN TRANSFORMER: hace que cada variable (numerica o categorica) pase por su pipeline correspondiente
preprocessor_polinomial = ColumnTransformer([
    ("num", numeric_transforme_polinomial, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# PIPELINE FINAL: preprocesamiento + modelo
pipeline_regresion_polinomial = Pipeline([
    ("dropper", dropper), # type: ignore
    ("preprocesamiento", preprocessor_polinomial),
    ("modelo", LinearRegression())
])

#BUSQUEDA DE HIPERPARAMETROS CON GRIDSEARCHV
param_grid = {
    "preprocesamiento__num__polynomial__degree": [1, 2, 3, 4]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42) #shuffle mezcla los datos, no lo hace en orden

grid = GridSearchCV(
    pipeline_regresion_polinomial,
    param_grid=param_grid,
    cv=cv,
    scoring="neg_root_mean_squared_error", #usar rmse negativo, minizando error
    n_jobs=-1
)

grid.fit(X_train, y_train) 



print("Mejor configuración:", grid.best_params_)
mejor_modelo = grid.best_estimator_

#METRICAS TRAIN
y_train_pred = mejor_modelo.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
r2_train = r2_score(y_train, y_train_pred)

print(f"Train RMSE: {rmse_train:.4f}")
print(f"Train MAE: {mae_train:.4f}")
print(f"Train R²: {r2_train:.4f}")

#METRICAS TEST
y_test_pred = mejor_modelo.predict(X_test)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"Test RMSE: {rmse_test:.4f}")
print(f"Test MAE: {mae_test:.4f}")
print(f"Test R²: {r2_test:.4f}")

Numéricas: ['Age', 'Weight (kg)', 'Height (m)', 'BMI', 'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)', 'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP', 'Estimated LDL (mg/dL)']
Categóricas: ['Sex', 'Smoking Status', 'Diabetes Status', 'Physical Activity Level', 'Family History of CVD', 'Blood Pressure Category']
Mejor configuración: {'preprocesamiento__num__polynomial__degree': 1}
Train RMSE: 10.7353
Train MAE: 3.5917
Train R²: 0.0645
Test RMSE: 10.6414
Test MAE: 3.4980
Test R²: 0.0529
